# TinyLlama Fine-Tuning





# 1. Install Libraries

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 10.1 MB/s eta 0:00:00


# 2. Load the Base Model (TinyLlama)

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.9.0+cu126
CUDA available: True


# 3. Prepare Your Dataset

In [ ]:
dataset = load_dataset("squad")
train_small = dataset["train"].select(range(50))
val_small = dataset["validation"].select(range(10))

train_small, val_small

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers'],
     num_rows: 50
 }),
 Dataset({
     features: ['id', 'title', 'context', 'question', 'answers'],
     num_rows: 10
 }))

# 4. Convert Each Row into a Prompt

In [ ]:
def format_item(example):
    answer = example["answers"]["text"][0] if example["answers"]["text"] else ""
    text = f"Context: {example['context']}\nQuestion: {example['question']}\nAnswer: {answer}"
    return {"text": text}

train_fmt = train_small.map(format_item)
val_fmt = val_small.map(format_item)

train_fmt[0]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]},
 'text': 'Context: Architecturally, the school has

# load the model

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True
)

model.resize_token_embeddings(len(tokenizer))

print("Loaded TinyLlama in 4-bit!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded TinyLlama in 4-bit!


# 5. Tokenize the Dataset

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_tok = train_fmt.map(tokenize, batched=True)
val_tok = val_fmt.map(tokenize, batched=True)

train_tok = train_tok.map(lambda x: {"labels": x["input_ids"]}, batched=True)
val_tok = val_tok.map(lambda x: {"labels": x["input_ids"]}, batched=True)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

# 6. Apply LoRA (4GB-friendly training)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


# 7. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="tinyllama_squad_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=50,
    bf16=False,
    fp16=False,
    save_total_limit=2,
)

# 8. Start Training

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer

/tmp/ipython-input-2770321885.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# 9. Save the Trained Model

In [ ]:
import wandb
wandb.init(project="tinyllama_squad_lora_finetune")
trainer.train()
trainer.save_model("tinyllama_squad_lora")
tokenizer.save_pretrained("tinyllama_squad_lora")

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: trivenirvp (trivenirvp-excelr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
5,2.174800
10,2.066400


('tinyllama_squad_lora/tokenizer_config.json',
 'tinyllama_squad_lora/special_tokens_map.json',
 'tinyllama_squad_lora/chat_template.jinja',
 'tinyllama_squad_lora/tokenizer.model',
 'tinyllama_squad_lora/added_tokens.json',
 'tinyllama_squad_lora/tokenizer.json')

# 10. Inference Code

In [ ]:
def ask(question, context):
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=60)
    return tokenizer.decode(output[0], skip_special_tokens=True)

sample = val_small[0]
ask(sample["question"], sample["context"])

'Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.\nQuestion: Which NFL team represented the AFC at Super Bowl 50?\nAnswer: The Denver Broncos represented the American Football Conference (AFC) at Super Bowl 50.'

In [ ]:
def ask(question, context="", max_new_tokens=80):
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
       do_sample=False,
        temperature=0.3
    )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    # remove the prompt from output
    answer = answer.replace(prompt, "").strip()

    print("Answer:", answer)


In [ ]:
ask("What is the capital of karnataka ?", "")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer: Bangalore is the capital of karnataka.


In [ ]:
def ask(question, context):
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=80)
    full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # REMOVE the prompt (paragraph + question)
    answer = full_output.replace(prompt, "").strip()

    return answer


In [ ]:
print(ask(question, context))


Bangalore

Sentence 2: The capital of Karnataka is Bangalore.

Sentence 3: The capital of Karnataka is Bangalore.

Sentence 4: The capital of Karnataka is Bangalore.

Sentence 5: The capital of Karnataka is Bang
